# 07: EOxElements in Jupyter

The EOxElements web components from the browser exercises run inside a Python
notebook via [`ipyeoxelements`](https://github.com/EOX-A/EOxElements-Jupyter)
(built on [anywidget](https://anywidget.dev/)). The map configuration is the *same dict*
you wrote as JSON in exercise 01.

| Python class | Web component |
| --- | --- |
| `EOxMap` | `<eox-map>` |
| `EOxLayercontrol` | `<eox-layercontrol>` |
| `EOxChart` | `<eox-chart>` |

## 1. A map from a Python dict

This is exercise 01's layer configuration written as a Python dict: an OpenStreetMap
base layer plus a Sentinel-2 GeoZarr layer read from S3, styled as a true-colour
RGB composite.

In [ ]:
from ipyeoxelements import EOxMap

zarr_url = (
    "https://s3.explorer.eopf.copernicus.eu/esa-zarr-sentinel-explorer-fra"
    "/tests-output/sentinel-2-l2a"
    "/S2B_MSIL2A_20260608T100029_N0512_R122_T33TVF_20260608T135005.zarr"
    "/measurements/reflectance"
)

layers = [
    {
        "type": "Tile",
        "properties": {"id": "osm", "title": "OpenStreetMap"},
        "source": {
            "type": "XYZ",
            "url": "https://tiles.maps.eox.at/wmts/1.0.0/osm_3857/default/g/{z}/{y}/{x}.jpg",
        },
    },
    {
        "type": "WebGLTile",
        "properties": {"id": "geozarr", "title": "Sentinel-2 GeoZarr"},
        "source": {
            "type": "GeoZarr",
            "url": zarr_url,
            "bands": ["b04", "b03", "b02"],
        },
        "style": {
            "gamma": 1.5,
            "color": [
                "color",
                ["interpolate", ["linear"], ["band", 1], 0, 0, 0.5, 255],
                ["interpolate", ["linear"], ["band", 2], 0, 0, 0.5, 255],
                ["interpolate", ["linear"], ["band", 3], 0, 0, 0.5, 255]
            ],
        },
    },
]

# layout height is required — the map element is height:100%, so without it the cell collapses.
eox_map = EOxMap(layers=layers, center=[14.09, 41.1], zoom=10, layout={"height": "500px"})
eox_map

## 2. Add a layer control

Link an `EOxLayercontrol` to the map with `for_` (note the trailing underscore:
`for` is a Python keyword) and place them side by side with an `ipywidgets` `HBox`.

In [ ]:
from ipywidgets import HBox
from ipyeoxelements import EOxMap, EOxLayercontrol

# In an HBox the children also need a width — otherwise the map collapses to 0 width
# and only the layer control shows.
linked_map = EOxMap(
    id="main-map",
    layers=layers,
    center=[14.09, 41.1],
    zoom=10,
    layout={"height": "500px", "width": "600px"},
)
control = EOxLayercontrol(
    for_="#main-map", layout={"height": "500px", "width": "320px"}
)

HBox([control, linked_map])

## 3. Band math: an NDVI vegetation index

The same `WebGLTile` GeoZarr layer can compute a spectral index from the raw bands.
Here we read NIR (`b08`) and Red (`b04`) and colour the Normalized Difference
Vegetation Index — vegetation shows up green, water and bare ground stay dark.

In [ ]:
from ipyeoxelements import EOxMap

# NDVI = (NIR - Red) / (NIR + Red). With a WebGLTile GeoZarr layer the index is
# computed on the GPU from the band values, then coloured with a ramp.
ndvi_layers = [
    {
        "type": "WebGLTile",
        "properties": {"id": "ndvi", "title": "NDVI"},
        "source": {
            "type": "GeoZarr",
            "url": zarr_url,
            "bands": ["b08", "b04"],  # band 1 = NIR (b08), band 2 = Red (b04)
        },
        "style": {
            "color": [
                "interpolate",
                ["linear"],
                [
                    "/",
                    ["-", ["band", 1], ["band", 2]],
                    ["+", ["band", 1], ["band", 2]],
                ],
                -0.1, [70, 90, 140],    # water / non-vegetated
                0.1, [180, 150, 110],   # bare soil
                0.4, [200, 210, 120],   # sparse vegetation
                0.7, [60, 150, 50],     # vegetation
                0.9, [0, 80, 0],        # dense vegetation
            ],
        },
    },
]

ndvi_map = EOxMap(
    layers=ndvi_layers, center=[14.09, 41.1], zoom=10, layout={"height": "500px"}
)
ndvi_map

## 4. A chart from the data

`EOxChart` renders a [Vega-Lite](https://vega.github.io/vega-lite/) spec. Here we read
the coarse 60 m overview of each RGB band **directly from the GeoZarr store** with
`zarr`, compute the mean surface reflectance per band, and chart it next to the map.

In [ ]:
import zarr
import numpy as np
from ipyeoxelements import EOxChart

# Read the coarse 60 m overview of each RGB band straight from the GeoZarr store.
# Each r60m band is a single 1830x1830 chunk, so this downloads ~13 MB per band.
r60m = zarr.open_group(zarr_url + "/r60m", mode="r")

bands = {"B04 (Red)": "b04", "B03 (Green)": "b03", "B02 (Blue)": "b02"}
values = []
for label, key in bands.items():
    arr = r60m[key][:]
    arr = arr[np.isfinite(arr) & (arr > 0)]  # drop nodata
    values.append({"band": label, "reflectance": round(float(arr.mean()), 4)})

spec = {
    "$schema": "https://vega.github.io/schema/vega-lite/v5.json",
    "mark": {"type": "bar", "tooltip": True, "fill": "#00ae9d"},
    "data": {"values": values},
    "encoding": {
        "x": {"field": "band", "type": "ordinal", "title": "Band"},
        "y": {"field": "reflectance", "type": "quantitative", "title": "Mean reflectance"},
    },
}

chart = EOxChart(spec=spec, layout={"height": "400px"})
chart